In [39]:
import pandas as pd
import re

file_path = '12.xlsx'
df = pd.read_excel(file_path)

# 컬럼명 확인 (이걸 먼저 봐야 함)
df.columns


c:\Users\KDS26\anaconda3\envs\bigdata\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Index(['번호', '개방서비스명', '개방서비스아이디', '개방자치단체코드', '관리번호', '인허가일자', '인허가취소일자',
       '영업상태구분코드', '영업상태명', '상세영업상태코드', '상세영업상태명', '폐업일자', '휴업시작일자', '휴업종료일자',
       '재개업일자', '소재지전화', '소재지면적', '소재지우편번호', '소재지전체주소', '도로명전체주소', '도로명우편번호',
       '사업장명', '최종수정시점', '데이터갱신구분', '데이터갱신일자', '업태구분명', '좌표정보X(EPSG5174)',
       '좌표정보Y(EPSG5174)', '축산업무구분명', '축산물가공업구분명', '축산일련번호', '권리주체일련번호',
       '총직원수'],
      dtype='object')

In [40]:
df.shape

(3473, 33)

In [41]:
meat = df[[
    '사업장명',
    '영업상태명',
    '소재지전체주소'
]].copy()


In [42]:
meat['영업상태명'].value_counts()


영업상태명
영업/정상    3473
Name: count, dtype: int64

In [50]:
def extract_admin(address):
    if pd.isna(address):
        return pd.Series([None, None, None, None])
    
    # 1️⃣ 시
    si_match = re.search(r'([가-힣]+광역시|[가-힣]+시)', address)
    si = si_match.group(1) if si_match else None
    
    # 2️⃣ 구 / 군 (시는 제외)
    sigungu_match = re.search(
        r'(?:광역시|시)\s+([가-힣]+(구|군))', address
    )
    sigungu = sigungu_match.group(1) if sigungu_match else None
    
    # 3️⃣ 읍 / 면 (있으면 동은 안 봄)
    eup_myeon_match = re.search(r'([가-힣]+(읍|면))', address)
    eup_myeon = eup_myeon_match.group(1) if eup_myeon_match else None
    
    # 4️⃣ 동 (읍/면이 없을 때만)
    dong = None
    if eup_myeon is None:
        dong_match = re.search(r'([가-힣]+동)', address)
        dong = dong_match.group(1) if dong_match else None
    
    return pd.Series([si, sigungu, eup_myeon, dong])

In [51]:
meat[['si', 'sigungu', 'eup_myeon', 'dong']] = (
    meat['소재지전체주소']
    .apply(extract_admin)
)

meat[['소재지전체주소', 'si', 'sigungu', 'eup_myeon', 'dong']].head()


,소재지전체주소,si,sigungu,eup_myeon,dong
0,대구광역시 달성군 논공읍 금포리 1313-3 102호,대구광역시,달성군,논공읍,None
1,대구광역시 달성군 논공읍 금포리 1313-3 102호,대구광역시,달성군,논공읍,None
2,대구광역시 달성군 논공읍 금포리 1313-3 101호,대구광역시,달성군,논공읍,None
3,대구광역시 달성군 논공읍 금포리 1313-3 101호,대구광역시,달성군,논공읍,None
4,대구광역시 중구 대봉동 14-21,대구광역시,중구,None,대봉동


In [53]:
meat[['si', 'sigungu', 'eup_myeon', 'dong']].head(30)

,si,sigungu,eup_myeon,dong
0,대구광역시,달성군,논공읍,None
1,대구광역시,달성군,논공읍,None
2,대구광역시,달성군,논공읍,None
3,대구광역시,달성군,논공읍,None
4,대구광역시,중구,None,대봉동
5,대구광역시,달서구,None,장기동
6,대구광역시,달성군,현풍읍,None
7,대구광역시,북구,None,서변동
8,대구광역시,북구,None,서변동
9,대구광역시,중구,None,남일동


In [54]:
# 동 지역만 따로 경쟁도 계산
dong_competition = (
    meat[meat['dong'].notna()]           # 동 지역만
    .groupby(['sigungu', 'dong'])
    .size()
    .reset_index(name='meat_shop_cnt')
)


In [55]:
# 고기집 많은 동 top 10
dong_competition.sort_values(
    'meat_shop_cnt', ascending=False
).head(10)


,sigungu,dong,meat_shop_cnt
0,남구,대명동,130
84,서구,비산동,85
89,서구,평리동,81
61,북구,구암동,79
73,북구,산격동,77
48,동구,신암동,71
65,북구,노원동,69
13,달서구,송현동,69
49,동구,신천동,65
58,북구,검단동,65


In [56]:
# 고기집 적은 동 top 10
dong_competition.sort_values(
    'meat_shop_cnt', ascending=True
).head(10)


,sigungu,dong,meat_shop_cnt
30,동구,대림동,1
29,동구,괴전동,1
36,동구,미대동,1
124,중구,서야동,1
127,중구,인교동,1
126,중구,수창동,1
125,중구,수동,1
119,중구,동문동,1
106,수성구,욱수동,1
105,수성구,연호동,1


In [58]:
# 읍/면 지역만 따로 경쟁도 계산
eup_competition = (
    meat[meat['eup_myeon'].notna()]       # 읍/면 지역만
    .groupby(['sigungu', 'eup_myeon'])
    .size()
    .reset_index(name='meat_shop_cnt')
)

# 고기집 ㅁㄶ은 읍 top 10
eup_competition.sort_values(
    'meat_shop_cnt', ascending=False
).head(10)


,sigungu,eup_myeon,meat_shop_cnt
15,달성군,화원읍,86
8,달성군,논공읍,84
9,달성군,다사읍,76
10,달성군,옥포읍,71
14,달성군,현풍읍,37
0,군위군,군위읍,19
6,달성군,가창면,13
12,달성군,유가읍,12
13,달성군,하빈면,10
5,군위군,효령면,9


In [59]:
# 고기집 적은 읍 top 10
eup_competition.sort_values(
    'meat_shop_cnt', ascending=True
).head(10)


,sigungu,eup_myeon,meat_shop_cnt
1,군위군,부계면,1
16,동구,한올면,1
3,군위군,우보면,2
2,군위군,소보면,2
11,달성군,유가면,3
4,군위군,의흥면,5
7,달성군,구지면,7
5,군위군,효령면,9
13,달성군,하빈면,10
12,달성군,유가읍,12


In [61]:
# 도심 동 vs 외곽 읍 비교

summary = pd.DataFrame({
    '지역유형': ['동', '읍/면'],
    '평균_고기집_수': [
        dong_competition['meat_shop_cnt'].mean(),
        eup_competition['meat_shop_cnt'].mean()
    ],
    '중앙값_고기집_수': [
        dong_competition['meat_shop_cnt'].median(),
        eup_competition['meat_shop_cnt'].median()
    ]
})

summary


,지역유형,평균_고기집_수,중앙값_고기집_수
0,동,23.325581,17.0
1,읍/면,25.764706,10.0


In [63]:
# 동 유망 후보
dong_candidates = dong_competition[
    dong_competition['meat_shop_cnt'] <= 
    dong_competition['meat_shop_cnt'].quantile(0.3)
]

# 읍 유망 후보
eup_candidates = eup_competition[
    eup_competition['meat_shop_cnt'] <= 
    eup_competition['meat_shop_cnt'].quantile(0.5)
]

dong_candidates.head()
eup_candidates.head()


,sigungu,eup_myeon,meat_shop_cnt
1,군위군,부계면,1
2,군위군,소보면,2
3,군위군,우보면,2
4,군위군,의흥면,5
5,군위군,효령면,9
